In [ ]:
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal, Optional
import json

# --- 1. CONFIGURACIÓN ---
# ¡OJO! Reemplaza esto con tu clave real
API_KEY = "pedirmela (Diego)" 

SYSTEM_PROMPT = """
Eres un Analista de Datos Experto para Amazon. Tu trabajo es extraer especificaciones técnicas estructuradas de títulos de productos para un modelo de Machine Learning (XGBoost).
Reglas:
1. Sé preciso con los números. Si dice '1TB', storage_gb es 1024.0.
2. Identifica el 'market_tier' basándote en la marca y palabras clave (Pro, Ultra, Basics, Lite).
3. Si el producto es un pack (ej. 2-Pack), indica pack_count=2.
4. Clasifica estrictamente en una de las 12 categorías permitidas.
5. SMART HOME: Timbres (doorbells) y cámaras de seguridad van en 'Networking & Smart Home'.
6. WEARABLES: Solo relojes y pulseras que se llevan puestos.
7. RESOLUCIÓN: Extrae 'resolution_standard' solo para pantallas o cámaras.
"""

client = OpenAI(api_key=API_KEY)

# --- 2. DEFINICIÓN DEL ESQUEMA (TU MODELO MAESTRO) ---
class AmazonFinalSpecs(BaseModel):
    # 1. Taxonomía (Sin cambios)
    category: Literal[
        "Smartphones & Wearables", "Tablets & E-Readers", "Laptops & Chromebooks",
        "Desktops, Workstations & Servers", "PC Components (Core)", "Peripherals & Input Devices",
        "Displays & Mounting", "Audio & Video Equipment", "Cameras & Photography",
        "Networking & Smart Home", "Gaming Systems & VR", "Accessories & Consumables"
    ]

    # 2. Anclas de Valor
    market_tier: Literal["Budget", "Mainstream", "Premium", "Enterprise/Professional"]
    condition: Literal["New", "Renewed/Refurbished"]
    is_premium_brand: bool
    tech_generation: Literal["Cutting-Edge", "Current-Gen", "Last-Gen", "Legacy"]

    # 3. Especificaciones Numéricas (Densas)
    ram_gb: Optional[float] = Field(None, description="RAM o VRAM del sistema en GB")
    storage_gb: Optional[float] = Field(None, description="Capacidad almacenamiento en GB")
    size_value: Optional[float] = Field(None, description="Pulgadas (Screens) o mm (Lentes/Audio)")
    
    # --- AQUÍ ESTÁ EL CAMBIO IMPORTANTE ---
    # Performance value sigue sirviendo para Hz, DPI, MP (Megapixeles)
    performance_value: Optional[float] = Field(None, description="Hz (Monitor), DPI (Mouse), MP (Cámara), Read Speed (MB/s)")
    power_wattage: Optional[float] = Field(None, description="Watios (W)")

    # 4. Descriptores Categóricos (NUEVO: RESOLUCIÓN)
    resolution_standard: Optional[Literal["HD/HD+", "FHD (1080p)", "2K/QHD (1440p)", "4K/UHD (2160p)", "5K/8K+"]] = Field(
        None, description="Estándar de resolución visual. Solo para pantallas, cámaras (video), monitores o proyectores."
    )
    
    cpu_gpu_tier: Optional[str] = Field(None, description="Ej: i7, Ryzen 5, RTX 4060, M3")
    connectivity: Optional[str] = Field(None, description="WiFi 6, 5G, Bluetooth, Wired, PoE")
    
    brand: str
    pack_count: int = Field(1, description="Número de unidades en el paquete")
    confidence: float = Field(description="Confianza en la extracción (0-1)")

# --- 3. DATOS DE PRUEBA (Muestra variada de tu archivo) ---
titulos_prueba = [
    # 1. Laptop con CPU y RAM específica (Prueba para Laptops & Chromebooks)
    "Lenovo Ideapad 3 Laptop, 15.6\" HD Touchscreen, 11th Gen Intel Core i3-1115G4, 12GB DDR4 RAM, 512GB PCIe SSD",
    
    # 2. Consola de Videojuegos (Prueba para Gaming Systems & VR)
    "Xbox Series X 1TB Gaming Console + 1 Wireless Controller - True 4K Gaming, Up to 120 FPS",
    
    # 3. Monitor de alta resolución y Hz (Prueba para Displays & Mounting + Resolution_standard)
    "ASUS ROG Strix 32” 4K HDR Gaming Monitor (XG32UCG) – Dual Mode (4K 160Hz/FHD 320Hz), 0.3ms, Fast IPS",
    
    # 4. Periférico Gaming (Prueba para Peripherals & Input Devices + DPI)
    "Logitech MX Master 3S for Business, Wireless Mouse with Quiet Clicks, 8K DPI, Bluetooth, USB-C",
    
    # 5. Equipo de Audio Profesional (Prueba para Audio & Video Equipment)
    "Sennheiser HD 600 - Audiophile Hi-Res Open Back Dynamic Headphone",
    
    # 6. Almacenamiento NAS / Servidor (Prueba para Desktops, Workstations & Servers)
    "Synology 2-Bay DiskStation DS224+ (Diskless)",
    
    # 7. Wearable real (Para contrastar con el timbre inteligente)
    "Xiaomi Smart Band 9 Global Version (2024) 1.62\" Amoled Display, 233 mAh Battery, BT 5.4",
    
    # 8. Impresora (Prueba para Accessories & Consumables o Peripherals según volumen)
    "Canon PIXMA TR8620a - All-in-One Printer Home Office | Copier | Scanner | Fax | Airprint",
    
    # 9. Lente de Cámara Profesional (Prueba para Cameras & Photography + mm)
    "Sony FE 16-35mm F2.8 GM II",
    
    # 10. Software / Juego físico (Prueba de Gaming Systems & VR)
    "Final Fantasy VII and Final Fantasy VIII Remastered - Twin Pack (Nintendo Switch)"
]

# --- 4. EJECUCIÓN ---
print("🧪 Iniciando prueba con 5 productos...\n")
resultados = []

for titulo in titulos_prueba:
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Analiza este título: {titulo}"}
            ],
            response_format=AmazonFinalSpecs,
        )
        
        datos = completion.choices[0].message.parsed.dict()
        datos['titulo_original'] = titulo[:30] + "..." # Acortar para visualizar
        resultados.append(datos)
        print(f"✅ Procesado: {datos['brand']} - {datos['category']}")
        
    except Exception as e:
        print(f"❌ Error con '{titulo}': {e}")

# --- 5. VISUALIZACIÓN ---
df_test = pd.DataFrame(resultados)

# Reordenar columnas para ver lo importante primero
cols = ['titulo_original', 'category', 'market_tier', 'ram_gb', 'storage_gb', 'power_wattage', 'pack_count', 'confidence']
df_test = df_test[cols + [c for c in df_test.columns if c not in cols]]

# Mostrar el DataFrame
df_test

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
🧪 Iniciando prueba con 5 productos...



C:\Users\diego\AppData\Local\Temp\ipykernel_3624\3817738026.py:111: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  datos = completion.choices[0].message.parsed.dict()


✅ Procesado: Lenovo - Laptops & Chromebooks
✅ Procesado: Xbox - Gaming Systems & VR
✅ Procesado: ASUS - Displays & Mounting
✅ Procesado: Logitech - Peripherals & Input Devices
✅ Procesado: Sennheiser - Audio & Video Equipment
✅ Procesado: Synology - Desktops, Workstations & Servers
✅ Procesado: Xiaomi - Smartphones & Wearables
✅ Procesado: Canon - Accessories & Consumables
✅ Procesado: Sony - Cameras & Photography
✅ Procesado: Square Enix - Gaming Systems & VR


,titulo_original,category,market_tier,ram_gb,storage_gb,power_wattage,pack_count,confidence,condition,is_premium_brand,tech_generation,size_value,performance_value,resolution_standard,cpu_gpu_tier,connectivity,brand
0,"Lenovo Ideapad 3 Laptop, 15.6""...",Laptops & Chromebooks,Mainstream,12.0,512.0,None,1,0.95,New,False,Current-Gen,15.60,NaN,HD/HD+,i3-1115G4,None,Lenovo
1,Xbox Series X 1TB Gaming Conso...,Gaming Systems & VR,Premium,NaN,1024.0,None,1,1.00,New,True,Current-Gen,NaN,NaN,4K/UHD (2160p),Unknown,Wired,Xbox
2,ASUS ROG Strix 32” 4K HDR Gami...,Displays & Mounting,Premium,NaN,NaN,None,1,1.00,New,True,Current-Gen,32.00,160.0,4K/UHD (2160p),None,None,ASUS
3,Logitech MX Master 3S for Busi...,Peripherals & Input Devices,Mainstream,NaN,NaN,None,1,1.00,New,True,Current-Gen,NaN,8000.0,None,None,"Bluetooth, USB-C",Logitech
4,Sennheiser HD 600 - Audiophile...,Audio & Video Equipment,Premium,NaN,NaN,None,1,0.95,New,True,Current-Gen,NaN,NaN,None,None,None,Sennheiser
5,Synology 2-Bay DiskStation DS2...,"Desktops, Workstations & Servers",Mainstream,NaN,NaN,None,2,0.90,New,True,Current-Gen,NaN,NaN,None,None,None,Synology
6,Xiaomi Smart Band 9 Global Ver...,Smartphones & Wearables,Mainstream,NaN,NaN,None,1,0.95,New,False,Current-Gen,1.62,NaN,None,None,Bluetooth 5.4,Xiaomi
7,Canon PIXMA TR8620a - All-in-O...,Accessories & Consumables,Mainstream,NaN,NaN,None,1,0.90,New,False,Current-Gen,NaN,NaN,None,None,Airprint,Canon
8,Sony FE 16-35mm F2.8 GM II...,Cameras & Photography,Premium,NaN,NaN,None,1,0.90,New,True,Current-Gen,NaN,NaN,None,None,None,Sony
9,Final Fantasy VII and Final Fa...,Gaming Systems & VR,Mainstream,NaN,NaN,None,2,0.95,New,False,Current-Gen,NaN,NaN,None,None,None,Square Enix


In [71]:
import pandas as pd


ev3 = pd.read_csv('amazon_specs_enriched.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
ev3['category'].value_counts()

category
Accessories & Consumables           2791
Audio & Video Equipment             1200
Peripherals & Input Devices         1063
Networking & Smart Home              491
Displays & Mounting                  462
Cameras & Photography                457
PC Components (Core)                 455
Laptops & Chromebooks                334
Desktops, Workstations & Servers     188
Smartphones & Wearables              154
Tablets & E-Readers                   78
Gaming Systems & VR                   41
ERROR_API_FAIL                         3
Name: count, dtype: int64

In [72]:
import pandas as pd
import numpy as np

# 1. Carga tus datos (ajusta los nombres de tus archivos)
df_specs = pd.read_csv('amazon_specs_enriched.csv')
df_prices = pd.read_csv('../../Datasets/evaluacion2.csv')

# 2. Une ambos datasets (usando el ID o el Título como llave)
df_final = pd.merge(
    df_specs, 
    df_prices, 
    left_on='original_title', 
    right_on='product_title'
)

# 3. Revertir el log1 para ver el precio real
# Si usaste log natural (ln(1+x)):
df_final['price_real'] = np.exp(df_final['log_original_price']) - 1

# Si usaste log base 10 (log10(1+x)):
# df['price_real'] = (10 ** df['log1_price']) - 1

# 4. Filtrar por la categoría Accesorios y ordenar por precio de mayor a menor
expensive_accessories = df_final[df_final['category'] == 'Peripherals & Input Devices'].sort_values(by='price_real', ascending=True)

# 5. Mostrar el Top 20
print("AUDITORÍA DE ACCESORIOS CAROS:")
expensive_accessories[['original_title', 'price_real', 'market_tier']].head(10)

AUDITORÍA DE ACCESORIOS CAROS:


,original_title,price_real,market_tier
634,"UGREEN USB A to USB B Printer Cable 5ft - High-Speed for HP, Canon, Brother, Samsung, Dell, Epson, Lexmark, Xerox, and More",5.69,Mainstream
192,"Amazon Basics 3-Button USB Wired Mouse with Scrolling and Tracking - Standard, Black",6.75,Budget
4196,"GE Blue Backlit Buttons Universal Remote Control, Samsung TV Remote Control Replacement, Samsung Remote Control for Smart TV, Roku Remote Replacement, Vizio, LG TV, Sony, 4-Device, Brushed Black 48843",7.23,Mainstream
2188,"Amazon Basics Square Mouse Pad, Cloth with Rubberized Base, Standard, Black, 12.4L x 10.6W inches",7.49,Budget
3698,"Verbatim Wired USB Computer Mouse - Corded USB Mouse for Laptops and PCs - Right or Left Hand Use, Blue 99743, 1.4"" x 2.4"" x 3.9""",7.95,Budget
4359,"Amazon Basics 3-Button USB Wired Mouse with Precision Scroll Wheel, Standard, Black",7.99,Budget
468,"Logitech B100 Wired Mouse for Computer and Laptop, USB Corded Mouse, Right or Left Hand Use - Black",7.99,Mainstream
5163,"C2G 6FT Premium Replacement AC Power Cord - Durable Power Cable for TV, Computer, Monitor, Appliance & More (24240), Pack of 1",7.99,Mainstream
5413,"RCA 3-Device Palm-Sized Universal Remote, Long Range IR, Replaces Most Major Remote Brands, Designed for Comfort, RCR503BE",8.36,Mainstream
1641,"Logitech Mouse Pad - Studio Series, Computer Mouse Mat with Anti-Slip Rubber Base, Easy Gliding, Spill-Resistant Surface, Durable Materials, Portable, in a Fresh Modern Design, Blue Grey",8.95,Mainstream


In [73]:
df_final['category'].value_counts()

category
Accessories & Consumables           2791
Audio & Video Equipment             1200
Peripherals & Input Devices         1063
Networking & Smart Home              491
Displays & Mounting                  462
Cameras & Photography                457
PC Components (Core)                 455
Laptops & Chromebooks                334
Desktops, Workstations & Servers     188
Smartphones & Wearables              154
Tablets & E-Readers                   78
Gaming Systems & VR                   41
ERROR_API_FAIL                         3
Name: count, dtype: int64